# Hybrid Semantic Search System - Evaluation & Analysis

This notebook provides comprehensive evaluation and analysis of the Hybrid Search System.

## Contents
1. Setup and Data Loading
2. Build Search Indices
3. Single Query Comparison
4. Batch Evaluation
5. Retriever Analysis
6. Performance Benchmarking
7. Result Visualization

## 1. Setup and Imports

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.insert(0, '../')

from src.search_engine import HybridSearchEngine
from src.utils import format_search_results

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful")

## 2. Initialize Search Engine

In [ ]:
# Configuration
DATA_PATH = "../data/semantic_search_dataset_2000.csv"
MODEL_DIR = "../models"

# Initialize engine
engine = HybridSearchEngine(
    data_path=DATA_PATH,
    model_dir=MODEL_DIR,
    device="cpu"
)

print("✅ Engine initialized")

## 3. Build Indices (First Time Only)

In [ ]:
# Build indices (this will take a few minutes on first run)
# Subsequent runs will load from disk
engine.build_indices()

print("\n✅ Indices built/loaded successfully")

## 4. Dataset Statistics

In [ ]:
# Get system info
system_info = engine.get_system_info()

print("Dataset Statistics:")
print(f"Total Documents: {system_info['data']['total_documents']}")
print(f"\nCategories:")
for cat, count in system_info['data']['categories'].items():
    print(f"  - {cat}: {count}")

print(f"\nDifficulty Levels:")
for diff, count in system_info['data']['difficulty_levels'].items():
    print(f"  - {diff}: {count}")

In [ ]:
# Visualize category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Categories
categories = system_info['data']['categories']
axes[0].bar(range(len(categories)), list(categories.values()))
axes[0].set_xticks(range(len(categories)))
axes[0].set_xticklabels(list(categories.keys()), rotation=45, ha='right')
axes[0].set_title('Document Distribution by Category')
axes[0].set_ylabel('Count')

# Difficulty
difficulty = system_info['data']['difficulty_levels']
axes[1].bar(range(len(difficulty)), list(difficulty.values()))
axes[1].set_xticks(range(len(difficulty)))
axes[1].set_xticklabels(list(difficulty.keys()))
axes[1].set_title('Document Distribution by Difficulty')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Single Query Comparison

In [ ]:
# Test query
test_query = "reverse singly linked list"

print(f"Query: '{test_query}'\n")
print("="*80)

# Compare all three methods
comparison = engine.compare_retrievers(test_query, top_k=5)

print("\nBM25 Results:")
for i, result in enumerate(comparison['bm25'], 1):
    print(f"{i}. {result['title']} (Score: {result['score']:.4f})")

print("\nSemantic Results:")
for i, result in enumerate(comparison['semantic'], 1):
    print(f"{i}. {result['title']} (Score: {result['score']:.4f})")

print("\nHybrid Results (RRF + Cross-Encoder):")
for i, result in enumerate(comparison['hybrid'], 1):
    print(f"{i}. {result['title']} (Score: {result['score']:.4f})")

print("\nOverlap Analysis:")
overlap = comparison['overlap_analysis']
print(f"BM25 ∩ Semantic: {overlap['bm25_semantic_overlap']}")
print(f"BM25 ∩ Hybrid: {overlap['bm25_hybrid_overlap']}")
print(f"Semantic ∩ Hybrid: {overlap['semantic_hybrid_overlap']}")
print(f"All Three: {overlap['all_three_overlap']}")

## 6. Batch Evaluation on Test Queries

In [ ]:
# Define test queries
test_queries = [
    "reverse singly linked list",
    "optimize SQL join performance",
    "deploy microservice on kubernetes",
    "reduce API latency in production",
    "implement binary search tree",
    "explain ACID properties in databases",
    "design scalable REST API",
    "kubernetes pod autoscaling",
    "database indexing strategies",
    "implement heap data structure"
]

print(f"Running evaluation on {len(test_queries)} queries...")

# Run evaluation
results = engine.evaluate_on_queries(
    test_queries,
    save_path="../outputs/evaluation_results.json"
)

print(f"\n✅ Evaluation complete!")
print(f"Average search time: {results['statistics']['avg_search_time']:.3f}s")

## 7. Analyze Results

In [ ]:
# Extract metrics for analysis
queries = []
bm25_top_scores = []
semantic_top_scores = []
hybrid_top_scores = []
search_times = []

for query_result in results['queries']:
    queries.append(query_result['query'])
    
    bm25_results = query_result['results']['bm25']
    semantic_results = query_result['results']['semantic']
    hybrid_results = query_result['results']['hybrid']
    
    bm25_top_scores.append(bm25_results[0]['score'] if bm25_results else 0)
    semantic_top_scores.append(semantic_results[0]['score'] if semantic_results else 0)
    hybrid_top_scores.append(hybrid_results[0]['score'] if hybrid_results else 0)
    
    search_times.append(query_result['search_time'])

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Query': queries,
    'BM25_Score': bm25_top_scores,
    'Semantic_Score': semantic_top_scores,
    'Hybrid_Score': hybrid_top_scores,
    'Search_Time': search_times
})

display(comparison_df)

In [ ]:
# Visualize score comparison
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(queries))
width = 0.25

ax.bar(x - width, bm25_top_scores, width, label='BM25', alpha=0.8)
ax.bar(x, semantic_top_scores, width, label='Semantic', alpha=0.8)
ax.bar(x + width, hybrid_top_scores, width, label='Hybrid', alpha=0.8)

ax.set_xlabel('Query Index')
ax.set_ylabel('Top Result Score')
ax.set_title('Top Result Scores Across Different Retrieval Methods')
ax.set_xticks(x)
ax.set_xticklabels(range(1, len(queries) + 1))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Search time analysis
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(range(len(search_times)), search_times, color='steelblue', alpha=0.7)
ax.axhline(y=np.mean(search_times), color='red', linestyle='--', 
           label=f'Average: {np.mean(search_times):.3f}s')
ax.set_xlabel('Query Index')
ax.set_ylabel('Search Time (seconds)')
ax.set_title('Hybrid Search Time per Query')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Search Time Statistics:")
print(f"  Mean: {np.mean(search_times):.3f}s")
print(f"  Median: {np.median(search_times):.3f}s")
print(f"  Std Dev: {np.std(search_times):.3f}s")
print(f"  Min: {np.min(search_times):.3f}s")
print(f"  Max: {np.max(search_times):.3f}s")

## 8. Detailed Analysis of Specific Query

In [ ]:
# Pick a specific query for detailed analysis
analysis_query = "optimize SQL join performance"

print(f"Detailed Analysis for: '{analysis_query}'\n")
print("="*80)

comparison = engine.compare_retrievers(analysis_query, top_k=5)

# Get unique documents across all methods
all_doc_ids = set()
for method in ['bm25', 'semantic', 'hybrid']:
    all_doc_ids.update(r['id'] for r in comparison[method])

print(f"Total unique documents retrieved: {len(all_doc_ids)}\n")

# Detailed comparison table
print("Rank Comparison:")
print(f"{'Doc ID':<40} {'BM25':<10} {'Semantic':<10} {'Hybrid':<10}")
print("-" * 70)

for doc_id in list(all_doc_ids)[:10]:  # Top 10 unique docs
    bm25_rank = next((i+1 for i, r in enumerate(comparison['bm25']) if r['id'] == doc_id), '-')
    sem_rank = next((i+1 for i, r in enumerate(comparison['semantic']) if r['id'] == doc_id), '-')
    hyb_rank = next((i+1 for i, r in enumerate(comparison['hybrid']) if r['id'] == doc_id), '-')
    
    print(f"{str(doc_id):<40} {str(bm25_rank):<10} {str(sem_rank):<10} {str(hyb_rank):<10}")

## 9. System Performance Summary

In [ ]:
print("="*80)
print("SYSTEM PERFORMANCE SUMMARY")
print("="*80)

system_info = engine.get_system_info()

print(f"\nDataset:")
print(f"  Total Documents: {system_info['data']['total_documents']}")
print(f"  Categories: {len(system_info['data']['categories'])}")
print(f"  Difficulty Levels: {len(system_info['data']['difficulty_levels'])}")

print(f"\nBM25 Index:")
print(f"  Vocabulary Size: {system_info['bm25']['vocabulary_size']}")
print(f"  Average Doc Length: {system_info['bm25']['avg_document_length']:.1f} tokens")

print(f"\nSemantic Index:")
print(f"  Embedding Dimension: {system_info['semantic']['embedding_dim']}")
print(f"  Index Type: {system_info['semantic']['index_type']}")
print(f"  Total Vectors: {system_info['semantic']['total_vectors']}")

print(f"\nSearch Performance:")
print(f"  Average Search Time: {results['statistics']['avg_search_time']:.3f}s")
print(f"  Queries Evaluated: {results['statistics']['total_queries']}")

print("\n" + "="*80)

## 10. Try Your Own Queries

In [ ]:
# Interactive query testing
def test_query(query_text, top_k=5):
    """
    Test a custom query and display results.
    """
    print(f"\nQuery: '{query_text}'")
    print("="*80)
    
    comparison = engine.compare_retrievers(query_text, top_k=top_k)
    
    print("\nHybrid Search Results:")
    for i, result in enumerate(comparison['hybrid'], 1):
        print(f"\n{i}. {result['title']}")
        print(f"   Category: {result['category']} | Difficulty: {result['difficulty']}")
        print(f"   Score: {result['score']:.4f}")
        print(f"   Preview: {result['body'][:150]}...")

# Example usage:
# test_query("How to implement a hash table in Python?")
# test_query("Docker container orchestration best practices")
# test_query("Graph traversal algorithms")

## Conclusion

This notebook demonstrated:
1. ✅ Building hybrid search indices
2. ✅ Comparing BM25, Semantic, and Hybrid retrieval
3. ✅ Evaluating search performance
4. ✅ Analyzing search results and overlap
5. ✅ Performance benchmarking

### Key Findings:
- **BM25** excels at exact keyword matching
- **Semantic Search** captures conceptual similarity
- **Hybrid Approach** combines strengths of both methods
- **Cross-Encoder Re-ranking** provides final precision boost

### Next Steps:
- Experiment with different queries
- Fine-tune RRF parameters (k value)
- Test on larger datasets
- Implement query expansion techniques
- Add user feedback loop for continuous improvement